# IA PARA CIENCIA DE DATOS

## Contenido

1. Cargar librerías
1. Leer archivo y procesar datos
1. ML methods
1. Fine-tuning
1. Model Comparison

## 1. Cargar librerías

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np # to use numpy arrays instead of lists
import pandas as pd # DataFrame (table)

# Matplotlib and seaborn for charting
import matplotlib.pyplot as plt # to plot
import seaborn as sns # to plot

np.random.seed(0)

In [2]:
# Sklearn processing modules
from sklearn import preprocessing # to normalize data
from sklearn.model_selection import train_test_split

In [3]:
# sklearn algorithms for ML
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn import svm
from sklearn.neighbors import KNeighborsClassifier

In [4]:
# Sklearn clustering algorithms
from sklearn.cluster import KMeans
from sklearn.cluster import MiniBatchKMeans
from sklearn.cluster import MeanShift

In [5]:
# Sklearn classification model evaluation metrics
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# Sklearn clustering model evaluation metrics
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

In [6]:
# Wilcoxon signed-rank test
from scipy.stats import wilcoxon
# α = 0.05
# Valor p ≤ α: La diferencia entre las medianas es significativamente diferente (Rechaza H0) 
# Valor p > α: La diferencia entre las medianas no es significativamente diferente (No puede rechazar H0) 

## 2. Leer archivo y procesar datos

### 2.1 Leer archivo

In [7]:
datos = pd.read_csv("archive.zip", compression='zip', header=0)
print(datos.head())
print()
print(datos.info())
print()
print(datos.nunique())

   size (cm)  shape  weight (g)  avg_price (₹)   color  taste     fruit_name
0       25.4  round      3089.2          137.1   green  sweet     watermelon
1       24.6  round      3283.9          163.8   green  sweet     watermelon
2        7.8  round       319.0           91.3   green  sweet  custard apple
3       20.0   oval      1607.0           85.7  orange  sweet         papaya
4       10.2   long       131.5           37.8  yellow  sweet         banana

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   size (cm)      10000 non-null  float64
 1   shape          10000 non-null  object 
 2   weight (g)     10000 non-null  float64
 3   avg_price (₹)  10000 non-null  float64
 4   color          10000 non-null  object 
 5   taste          10000 non-null  object 
 6   fruit_name     10000 non-null  object 
dtypes: float64(3), object(4)
memo

### 2.2 Preprocesamiento

In [8]:
def encodeCategoricalColumns(dataSet, columnsNames):
    enc = preprocessing.OrdinalEncoder()
    dataSet[columnsNames] = enc.fit_transform(dataSet[columnsNames])

    print(dataSet.head())
    print()
    print(dataSet.info())
    
    return dataSet

In [9]:
def scaleData(dataSet):
    """
    Normalize and smooth data
    """
    columnsNames = dataSet.columns[:-1]  # Exclude the last column (assumed to be the target variable)
    dataSet = dataSet.fillna(method='ffill')

    # fit the scaler
    scaler = preprocessing.StandardScaler()
    scaler.fit(dataSet[columnsNames])

    # Transform the data using the fitted scaler
    np_dataSet = scaler.transform(dataSet[columnsNames])

    # Convert the numpy array back to a DataFrame
    dataSet[columnsNames] = pd.DataFrame(np_dataSet, columns=columnsNames, index=dataSet.index)

    return dataSet

In [10]:
# Encode categorical variables
columnsNames = ['shape', 'color', 'taste', 'fruit_name']
dataEncoded = encodeCategoricalColumns(datos, columnsNames)
print()

# Scale data
dataScaled = scaleData(dataEncoded)
print(dataScaled.head())

   size (cm)  shape  weight (g)  avg_price (₹)  color  taste  fruit_name
0       25.4    2.0      3089.2          137.1    2.0    1.0        19.0
1       24.6    2.0      3283.9          163.8    2.0    1.0        19.0
2        7.8    2.0       319.0           91.3    2.0    1.0         5.0
3       20.0    1.0      1607.0           85.7    3.0    1.0        13.0
4       10.2    0.0       131.5           37.8    7.0    1.0         1.0

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   size (cm)      10000 non-null  float64
 1   shape          10000 non-null  float64
 2   weight (g)     10000 non-null  float64
 3   avg_price (₹)  10000 non-null  float64
 4   color          10000 non-null  float64
 5   taste          10000 non-null  float64
 6   fruit_name     10000 non-null  float64
dtypes: float64(7)
memory usage: 547.0 KB
None

   size (c

### 2.3 Split Data

In [11]:
# Split data into train and test sets
train_size = 0.75
test_size = 1 - train_size
trainSet, testSet = train_test_split(dataScaled, test_size=test_size, shuffle=True, random_state=0)

print("Train set")
print(trainSet.head())
print()
print('trainSet shape: ', trainSet.shape)
print()

print("Test set")
print(testSet.head())
print()
print('testSet shape: ', testSet.shape)

Train set
      size (cm)     shape  weight (g)  avg_price (₹)     color    taste  \
2967  -0.004784  0.929913   -0.211121       0.430949 -0.744021 -0.32263   
700   -0.473783  0.929913   -0.417926       0.230666  1.094090 -0.32263   
3481  -0.239284  0.929913   -0.289989       1.321949  1.094090 -0.32263   
1621  -0.864616 -0.765153   -0.594661      -0.372750  1.094090 -0.32263   
800   -0.129850 -0.765153   -0.369813      -0.046648 -0.744021 -0.32263   

      fruit_name  
2967         5.0  
700          0.0  
3481        17.0  
1621        18.0  
800         14.0  

trainSet shape:  (7500, 7)

Test set
      size (cm)     shape  weight (g)  avg_price (₹)     color     taste  \
9394   0.010850 -0.765153   -0.364756      -0.051784 -0.744021 -0.322630   
898   -0.520683 -0.765153   -0.522354       0.769888 -1.203549  1.780567   
2398  -0.817716 -0.765153   -0.609149      -1.266318  1.094090 -0.322630   
5906   0.073383 -0.765153   -0.282198       1.306543  1.553618 -0.322630   
2343  -

## 3. ML methods

### 3.1 Métodos clasificación

In [12]:
# DataFrames to store model predictions
columnsNames = ['fruit_name', 'Log Reg fruit_name', 'DT fruit_name', 'RF fruit_name', 'SVM fruit_name', 'KNN fruit_name', 'KMeans Cluster', 'MiniBatch KMeans Cluster', 'MeanShift Cluster']
trainModel = pd.DataFrame(columns=columnsNames)
testModel = pd.DataFrame(columns=columnsNames)
trainModel['fruit_name'] = trainSet['fruit_name']
testModel['fruit_name'] = testSet['fruit_name']
print(trainModel.head())
print()
print(testModel.head())

      fruit_name Log Reg fruit_name DT fruit_name RF fruit_name  \
2967         5.0                NaN           NaN           NaN   
700          0.0                NaN           NaN           NaN   
3481        17.0                NaN           NaN           NaN   
1621        18.0                NaN           NaN           NaN   
800         14.0                NaN           NaN           NaN   

     SVM fruit_name KNN fruit_name KMeans Cluster MiniBatch KMeans Cluster  \
2967            NaN            NaN            NaN                      NaN   
700             NaN            NaN            NaN                      NaN   
3481            NaN            NaN            NaN                      NaN   
1621            NaN            NaN            NaN                      NaN   
800             NaN            NaN            NaN                      NaN   

     MeanShift Cluster  
2967               NaN  
700                NaN  
3481               NaN  
1621               NaN  
800

In [13]:
# DataFrames to store model performance metrics
methodsUsed=['Log Reg', 'DT', 'RF', 'SVM', 'KNN', 'KMeans', 'MiniBatch KMeans', 'MeanShift']
performanceHeaders=['accuracy', 'precision','recall','f1-score']
trainModelMetrics = pd.DataFrame(index=methodsUsed, columns=performanceHeaders)
testModelMetrics = pd.DataFrame(index=methodsUsed, columns=performanceHeaders)
print(trainModelMetrics)
print()
print(testModelMetrics)

                 accuracy precision recall f1-score
Log Reg               NaN       NaN    NaN      NaN
DT                    NaN       NaN    NaN      NaN
RF                    NaN       NaN    NaN      NaN
SVM                   NaN       NaN    NaN      NaN
KNN                   NaN       NaN    NaN      NaN
KMeans                NaN       NaN    NaN      NaN
MiniBatch KMeans      NaN       NaN    NaN      NaN
MeanShift             NaN       NaN    NaN      NaN

                 accuracy precision recall f1-score
Log Reg               NaN       NaN    NaN      NaN
DT                    NaN       NaN    NaN      NaN
RF                    NaN       NaN    NaN      NaN
SVM                   NaN       NaN    NaN      NaN
KNN                   NaN       NaN    NaN      NaN
KMeans                NaN       NaN    NaN      NaN
MiniBatch KMeans      NaN       NaN    NaN      NaN
MeanShift             NaN       NaN    NaN      NaN


In [14]:
# DataFrame to store model performance metrics
methods = ['KMeans Cluster', 'MiniBatch KMeans Cluster', 'MeanShift Cluster']
performanceHeaders = ['Silhouette', 'Calinski-Harabasz', 'Davies-Bouldin']
trainModelMetrics2 = pd.DataFrame(columns=performanceHeaders, index=methods)
testModelMetrics2 = pd.DataFrame(columns=performanceHeaders, index=methods)
print(trainModelMetrics2)
print()
print(testModelMetrics2)

                         Silhouette Calinski-Harabasz Davies-Bouldin
KMeans Cluster                  NaN               NaN            NaN
MiniBatch KMeans Cluster        NaN               NaN            NaN
MeanShift Cluster               NaN               NaN            NaN

                         Silhouette Calinski-Harabasz Davies-Bouldin
KMeans Cluster                  NaN               NaN            NaN
MiniBatch KMeans Cluster        NaN               NaN            NaN
MeanShift Cluster               NaN               NaN            NaN


In [15]:
def computeModel(model, modelName, trainModel, testModel, trainSet, testSet):
    X_train = trainSet.iloc[:,:-1]
    y_train = trainSet.iloc[:,-1]
    X_test = testSet.iloc[:,:-1]

    model.fit(X_train, y_train)

    trainModel[modelName] = model.predict(X_train)
    testModel[modelName] = model.predict(X_test)
    
    return trainModel, testModel

In [16]:
def compareMethod(position, modelName, trainModelMetrics, testModelMetrics, trainModel, testModel):
    # --- Train Metrics ---
    trainModelMetrics.iloc[position, 0] = accuracy_score(trainModel['fruit_name'], trainModel[modelName])
    trainModelMetrics.iloc[position, 1] = precision_score(trainModel['fruit_name'], trainModel[modelName], average='macro', zero_division=0)
    trainModelMetrics.iloc[position, 2] = recall_score(trainModel['fruit_name'], trainModel[modelName], average='macro', zero_division=0)
    trainModelMetrics.iloc[position, 3] = f1_score(trainModel['fruit_name'], trainModel[modelName], average='macro', zero_division=0)

    # --- Test Metrics ---
    testModelMetrics.iloc[position, 0] = accuracy_score(testModel['fruit_name'], testModel[modelName])
    testModelMetrics.iloc[position, 1] = precision_score(testModel['fruit_name'], testModel[modelName ], average='macro', zero_division=0)
    testModelMetrics.iloc[position, 2] = recall_score(testModel['fruit_name'], testModel[modelName], average='macro', zero_division=0)
    testModelMetrics.iloc[position, 3] = f1_score(testModel['fruit_name'], testModel[modelName], average='macro', zero_division=0)

    return trainModelMetrics, testModelMetrics

In [17]:
def compareMethod2(position, columnsNames, modelName, trainModelMetrics, testModelMetrics, trainModel, testModel, trainSet, testSet):
    """
    Compare clustering methods using evaluation metrics
    """
    # Train set metrics
    trainModelMetrics.iloc[position, 0] = silhouette_score(trainSet[columnsNames], trainModel[modelName])
    trainModelMetrics.iloc[position, 1] = calinski_harabasz_score(trainSet[columnsNames], trainModel[modelName])
    trainModelMetrics.iloc[position, 2] = davies_bouldin_score(trainSet[columnsNames], trainModel[modelName])


    # Test set metrics
    testModelMetrics.iloc[position, 0] = silhouette_score(testSet[columnsNames], testModel[modelName])
    testModelMetrics.iloc[position, 1] = calinski_harabasz_score(testSet[columnsNames], testModel[modelName])
    testModelMetrics.iloc[position, 2] = davies_bouldin_score(testSet[columnsNames], testModel[modelName])

    return trainModelMetrics, testModelMetrics


In [18]:
# Perform classification with Logistic Regression
print('Classification with Logistic Regression')
modelName = 'Log Reg fruit_name'
postion = 0
classifier_log_reg = LogisticRegression()
trainModel, testModel = computeModel(classifier_log_reg, modelName, trainModel, testModel, trainSet, testSet)

print(testModel)

trainModelMetrics, testModelMetrics = compareMethod(postion, modelName, trainModelMetrics, testModelMetrics, trainModel, testModel)

print(trainModelMetrics)
print()
print(testModelMetrics)
print()

# Detailed classification report
report = classification_report(trainModel['fruit_name'], trainModel[modelName])
print("Detailed Classification Report for Decision Tree:")
print(report)

Classification with Logistic Regression


      fruit_name  Log Reg fruit_name DT fruit_name RF fruit_name  \
9394        14.0                14.0           NaN           NaN   
898          9.0                 9.0           NaN           NaN   
2398         3.0                 3.0           NaN           NaN   
5906        11.0                11.0           NaN           NaN   
2343         8.0                 8.0           NaN           NaN   
...          ...                 ...           ...           ...   
8764        15.0                15.0           NaN           NaN   
4359        12.0                12.0           NaN           NaN   
2041        16.0                16.0           NaN           NaN   
1108        15.0                15.0           NaN           NaN   
3332         3.0                 3.0           NaN           NaN   

     SVM fruit_name KNN fruit_name KMeans Cluster MiniBatch KMeans Cluster  \
9394            NaN            NaN            NaN                      NaN   
898             NaN        

In [19]:
# Perform classification with DT
print('Classification with DT')
modelName = 'DT fruit_name'
postion = 1
classifier_dt = DecisionTreeClassifier(max_depth=5)
trainModel, testModel = computeModel(classifier_dt, modelName, trainModel, testModel, trainSet, testSet)

print(testModel)

trainModelMetrics, testModelMetrics = compareMethod(postion, modelName, trainModelMetrics, testModelMetrics, trainModel, testModel)

print(trainModelMetrics)
print()
print(testModelMetrics)
print()

# Detailed classification report
report = classification_report(trainModel['fruit_name'], trainModel[modelName])
print("Detailed Classification Report for Decision Tree:")
print(report)

Classification with DT
      fruit_name  Log Reg fruit_name  DT fruit_name RF fruit_name  \
9394        14.0                14.0           14.0           NaN   
898          9.0                 9.0           14.0           NaN   
2398         3.0                 3.0           14.0           NaN   
5906        11.0                11.0           17.0           NaN   
2343         8.0                 8.0           14.0           NaN   
...          ...                 ...            ...           ...   
8764        15.0                15.0           15.0           NaN   
4359        12.0                12.0           14.0           NaN   
2041        16.0                16.0           14.0           NaN   
1108        15.0                15.0           15.0           NaN   
3332         3.0                 3.0           14.0           NaN   

     SVM fruit_name KNN fruit_name KMeans Cluster MiniBatch KMeans Cluster  \
9394            NaN            NaN            NaN                     

In [20]:
# Perform the class with RF
print('Classification with RF')
modelName = 'RF fruit_name'
postion = 2
classifier_rf = RandomForestClassifier(n_estimators=100, max_depth=5)
trainModel, testModel = computeModel(classifier_rf, modelName, trainModel, testModel, trainSet, testSet)

print(testModel)

trainModelMetrics, testModelMetrics = compareMethod(postion, modelName, trainModelMetrics, testModelMetrics, trainModel, testModel)

print(trainModelMetrics)
print()
print(testModelMetrics)
print()

# Detailed classification report
report = classification_report(trainModel['fruit_name'], trainModel[modelName])
print("Detailed Classification Report for Random Forest:")
print(report)

Classification with RF
      fruit_name  Log Reg fruit_name  DT fruit_name  RF fruit_name  \
9394        14.0                14.0           14.0           14.0   
898          9.0                 9.0           14.0            9.0   
2398         3.0                 3.0           14.0            3.0   
5906        11.0                11.0           17.0           11.0   
2343         8.0                 8.0           14.0            8.0   
...          ...                 ...            ...            ...   
8764        15.0                15.0           15.0           15.0   
4359        12.0                12.0           14.0           12.0   
2041        16.0                16.0           14.0           16.0   
1108        15.0                15.0           15.0           15.0   
3332         3.0                 3.0           14.0            3.0   

     SVM fruit_name KNN fruit_name KMeans Cluster MiniBatch KMeans Cluster  \
9394            NaN            NaN            NaN         

In [21]:
# Perform the class with SVM
print('Classification with SVM')
modelName = 'SVM fruit_name'
postion = 3
classifier_svm = svm.SVC()
trainModel, testModel = computeModel(classifier_svm, modelName, trainModel, testModel, trainSet, testSet)

print(testModel)

trainModelMetrics, testModelMetrics = compareMethod(postion, modelName, trainModelMetrics, testModelMetrics, trainModel, testModel)

print(trainModelMetrics)
print()
print(testModelMetrics)
print()

# Detailed classification report
report = classification_report(trainModel['fruit_name'], trainModel[modelName])
print("Detailed Classification Report for SVM:")
print(report)

Classification with SVM
      fruit_name  Log Reg fruit_name  DT fruit_name  RF fruit_name  \
9394        14.0                14.0           14.0           14.0   
898          9.0                 9.0           14.0            9.0   
2398         3.0                 3.0           14.0            3.0   
5906        11.0                11.0           17.0           11.0   
2343         8.0                 8.0           14.0            8.0   
...          ...                 ...            ...            ...   
8764        15.0                15.0           15.0           15.0   
4359        12.0                12.0           14.0           12.0   
2041        16.0                16.0           14.0           16.0   
1108        15.0                15.0           15.0           15.0   
3332         3.0                 3.0           14.0            3.0   

      SVM fruit_name KNN fruit_name KMeans Cluster MiniBatch KMeans Cluster  \
9394            14.0            NaN            NaN      

In [22]:
# Perform the class with KNN
print('Classification with KNN')
modelName = 'KNN fruit_name'
postion = 4
classifier_knn = KNeighborsClassifier(n_neighbors=2)
trainModel, testModel = computeModel(classifier_knn, modelName, trainModel, testModel, trainSet, testSet)
print(testModel)

trainModelMetrics, testModelMetrics = compareMethod(postion, modelName, trainModelMetrics, testModelMetrics, trainModel, testModel)

print(trainModelMetrics)
print()
print(testModelMetrics)
print()

# Detailed classification report
report = classification_report(trainModel['fruit_name'], trainModel[modelName])
print("Detailed Classification Report for KNN:")
print(report)

Classification with KNN
      fruit_name  Log Reg fruit_name  DT fruit_name  RF fruit_name  \
9394        14.0                14.0           14.0           14.0   
898          9.0                 9.0           14.0            9.0   
2398         3.0                 3.0           14.0            3.0   
5906        11.0                11.0           17.0           11.0   
2343         8.0                 8.0           14.0            8.0   
...          ...                 ...            ...            ...   
8764        15.0                15.0           15.0           15.0   
4359        12.0                12.0           14.0           12.0   
2041        16.0                16.0           14.0           16.0   
1108        15.0                15.0           15.0           15.0   
3332         3.0                 3.0           14.0            3.0   

      SVM fruit_name  KNN fruit_name KMeans Cluster MiniBatch KMeans Cluster  \
9394            14.0            14.0            NaN    

In [23]:
columnsNames = trainSet.columns[:-1]
print(columnsNames)

Index(['size (cm)', 'shape', 'weight (g)', 'avg_price (₹)', 'color', 'taste'], dtype='object')


In [24]:
modelName = 'KMeans Cluster'
position = 5
KMestimator = KMeans(n_clusters=20, random_state=0)
KMestimator.fit(trainSet[columnsNames])
# we obtain the cluster labels from the training set
trainModel[modelName]=KMestimator.labels_
print(trainModel.head())
print()

# we obtain the cluster labels for the test set
testModel[modelName] = KMestimator.predict(testSet[columnsNames])
print(testModel.head())
print()

position = 0
trainModelMetrics2, testModelMetrics2 = compareMethod2(position, columnsNames, modelName, trainModelMetrics2, testModelMetrics2, trainModel, testModel, trainSet, testSet)

print(trainModelMetrics2)
print()
print(testModelMetrics2)

      fruit_name  Log Reg fruit_name  DT fruit_name  RF fruit_name  \
2967         5.0                 5.0           17.0            5.0   
700          0.0                 0.0           14.0            0.0   
3481        17.0                17.0           17.0           17.0   
1621        18.0                18.0           14.0           18.0   
800         14.0                14.0           14.0           14.0   

      SVM fruit_name  KNN fruit_name  KMeans Cluster MiniBatch KMeans Cluster  \
2967             5.0             5.0               2                      NaN   
700              0.0             0.0              10                      NaN   
3481            17.0            17.0              10                      NaN   
1621            18.0            18.0               1                      NaN   
800             14.0            14.0              12                      NaN   

     MeanShift Cluster  
2967               NaN  
700                NaN  
3481             

In [25]:
# Perform MiniBatchKMeans clustering
modelName = 'MiniBatch KMeans Cluster'
position = 6
MBKestimator = MiniBatchKMeans(n_clusters=20, random_state=0)
MBKestimator.fit(trainSet[columnsNames])
# we obtain the cluster labels from the training set
trainModel[modelName]=MBKestimator.labels_
print(trainModel)
print()

# we obtain the cluster labels for the test set
testModel[modelName] = MBKestimator.predict(testSet[columnsNames])
print(testModel)
print()

position = 1
trainModelMetrics2, testModelMetrics2 = compareMethod2(position, columnsNames, modelName, trainModelMetrics2, testModelMetrics2, trainModel, testModel, trainSet, testSet)


print(trainModelMetrics2)
print()
print(testModelMetrics2)

      fruit_name  Log Reg fruit_name  DT fruit_name  RF fruit_name  \
2967         5.0                 5.0           17.0            5.0   
700          0.0                 0.0           14.0            0.0   
3481        17.0                17.0           17.0           17.0   
1621        18.0                18.0           14.0           18.0   
800         14.0                14.0           14.0           14.0   
...          ...                 ...            ...            ...   
9225        10.0                10.0           14.0           10.0   
4859        18.0                18.0           14.0           18.0   
3264         2.0                 2.0           14.0            2.0   
9845         2.0                 2.0           14.0            2.0   
2732        11.0                11.0           17.0           11.0   

      SVM fruit_name  KNN fruit_name  KMeans Cluster  \
2967             5.0             5.0               2   
700              0.0             0.0           

In [31]:
print(trainModel.nunique())
print()

trainClassValues = pd.DataFrame(trainModel['fruit_name'].value_counts().sort_index().values, columns=['fruit_name Count'])
trainClassValues['Log Reg fruit_name Count'] = trainModel['Log Reg fruit_name'].value_counts().sort_index()
trainClassValues['RF fruit_name Count'] = trainModel['RF fruit_name'].value_counts().sort_index()
print(trainClassValues)
print()

trainClusterValues = pd.DataFrame(trainModel['KMeans Cluster'].value_counts().sort_index().values, columns=['KMeans Cluster Count'])
trainClusterValues['MiniBatch KMeans Cluster Count'] = trainModel['MiniBatch KMeans Cluster'].value_counts().sort_index()
print(trainClusterValues)
print()

print(testModel.nunique())

fruit_name                  20
Log Reg fruit_name          20
DT fruit_name                8
RF fruit_name               20
SVM fruit_name              20
KNN fruit_name              20
KMeans Cluster              20
MiniBatch KMeans Cluster    20
MeanShift Cluster            0
dtype: int64

    fruit_name Count  Log Reg fruit_name Count  RF fruit_name Count
0                350                       350                  350
1                367                       367                  367
2                361                       361                  361
3                370                       370                  370
4                374                       374                  374
5                395                       395                  395
6                385                       385                  385
7                387                       387                  387
8                357                       357                  357
9                368       